In [1]:
import sys
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
sys.path.insert(0, str(Path('.')))
import torch
import pandas as pd
from modules.preprocessing import load_config, load_dataset, OUTPUT_NAMES, INPUT_COLS
from modules.training import run_iterative_training, plot_results, save_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

TARGET = 'Du'

Device: cuda


In [2]:
YAML_PATH = Path('hyperparameters.yaml')
hp, DS, DNN_DEFAULTS, OUTS, TRAIN_SPLIT, R2_THRESHOLD, INCREMENT, MAX_ITER = load_config(YAML_PATH)
RUN_TIMESTAMP = datetime.now(ZoneInfo('America/Chicago')).strftime('%Y%m%d_%H%M%S')
print(f'Target: {TARGET}  |  R2_threshold={R2_THRESHOLD}  increment={INCREMENT}  max_iter={MAX_ITER}')
print(f'Run timestamp: {RUN_TIMESTAMP}')

Target: Du  |  R2_threshold=0.9  increment=2000  max_iter=40
Run timestamp: 20260617_084751


In [3]:
DATA_PATH = Path('../1. Dataset/DNN_dataset_cleaned.csv')
X_all, Y_all = load_dataset(DATA_PATH, INPUT_COLS, OUTPUT_NAMES)
print(f'Full dataset pool: X={X_all.shape}  |  Y={Y_all.shape}')

Loaded 110,649 rows x 42 columns
Full dataset pool: X=(110649, 23)  |  Y=(110649, 8)


In [ ]:
result = run_iterative_training(
    TARGET, X_all, Y_all, TRAIN_SPLIT,
    INCREMENT, MAX_ITER, R2_THRESHOLD,
    OUTPUT_NAMES, OUTS, DNN_DEFAULTS, device,
    log_path=Path('logs') / f'train_{TARGET}_{RUN_TIMESTAMP}.out',
    checkpoint_dir=Path('checkpoints'),
)

Log file: logs/train_Du_20260617_084751.out

  TRAINING FOR OUTPUT: Du

  ITERATION   1 | TOTAL SAMPLES=  2,000 | TRAIN=  1,600 | TEST=    400

     Epoch     Train MSE      Test MSE   Train R2   Test R2          Time
    ------  ------------  ------------  ---------  --------  ------------
       500      0.631159      0.521578     0.4702    0.4430           27s


In [ ]:
print(f"\nBest model for {TARGET}")
print(f"  Total subset : {result['final_n']:,}")
print(f"  Train rows   : {result['train_n']:,}")
print(f"  Test rows    : {result['test_n']:,}")
print(f"  Train R2     : {result['train_r2']:.4f}")
print(f"  Test R2      : {result['test_r2']:.4f}")
print(f"  R2 >= 0.90   : {'YES' if result['test_r2'] >= R2_THRESHOLD else 'NO'}")

In [ ]:
plot_results(TARGET, result, R2_THRESHOLD, save_path=f'results_{TARGET}.png')

In [ ]:
save_model(result['model'], TARGET, Path('saved_models'))